# Camptocamp.org Scraper — Climbing Routes

This notebook retrieves climbing routes and their characteristics from
the public API of [camptocamp.org](https://www.camptocamp.org), to
create a dataset suitable for analysis.

## 1. Library imports and settings configuration

In [ ]:
import csv
import os
import time
import requests
import pandas as pd
import math

# API endpoint root
BASE_URL = "https://api.camptocamp.org" 

# Python constant that will be used to assign a value to an API parameter (language of the results)
LANG = "fr" 

# Python constant that will be used to assign a value to an API parameter (number of rows per page)
PAGE_SIZE = 100 

# Pause between each network call (in seconds), to avoid overloading the API
SLEEP_BETWEEN_CALLS = 0.3

# Columns that will be exported in the final CSV file
FIELDS = [
    "document_id", "title", "title_prefix", "quality", "activities",
    "climbing_outdoor_type", "global_rating", "engagement_rating",
    "risk_rating", "equipment_rating", "exposition_rock_rating",
    "rock_free_rating", "rock_required_rating", "aid_rating",
    "rock_types", "orientations", "configuration",
    "elevation_min", "elevation_max", "height_diff_up", "height_diff_down",
    "height_diff_difficulties", "durations", "description",
]

print("Configuration chargée.")


Configuration chargée.


## 2. Retrieval and Parsing Functions

### 2.1 Side functions

In [ ]:
# Retrieves a page from the list of climbing routes
def fetch_routes_page(offset, limit=PAGE_SIZE):
    params = {"act": "rock_climbing", "limit": limit, "offset": offset, "pl": LANG} # definition of the API parameters for the request
    resp = requests.get(f"{BASE_URL}/routes", params=params, timeout=30) # sends a GET request to the API to retrieve the list of climbing routes with the defined parameters
                                                                         # concretely, the function transforms the f-string into a complete URL, then sends a GET request to this URL with the defined parameters. 
                                                                         # The timeout=30 parameter means that the request will be cancelled if it takes more than 30 seconds.
    resp.raise_for_status() # Check the nature of the API response. This response contains an HTTP status code. If the status code indicates an error (for example, 4xx or 5xx), this method raises an HTTPError exception. This allows for appropriate error handling in the calling code.
    return resp.json() # If there is no error, return the content of the response as a Python dictionary (the response is in JSON format, and the json() method converts it into a Python dictionary).

# Join a list with ';', or return None if absent/empty.
def join_or_none(values):
    if not values:
        return None
    return ";".join(values)

# Retrieves the detailed document of a climbing route (/routes/{id}). This allows to obtain the parameters "description", "configuration" and "rock_types".
def fetch_route_detail(document_id):

    url = f"{BASE_URL}/routes/{document_id}" # API endpoint to retrieve the details of a specific climbing route, identified by its document_id.
    resp = requests.get(url, params={"pl": LANG}, timeout=30)
    resp.raise_for_status()
    data = resp.json() # Saving the response content as a Python dictionary.

    description = None # Initialize the “description” variable to None. This serves as the default value in case no description is found in the locales.
    locales = data.get("locales", []) # locales is a list of dictionaries, each representing a localized (translated) version of the route information. 
                                      # Here, we retrieve the list of locales from the data dictionary. If the key "locales" does not exist, we get an empty list by default.
    if locales:
        for loc in locales:
            if loc.get("lang") == LANG: # taking for the locale corresponding to the language defined by the constant LANG (here, "fr" for French).
                description = loc.get("description") # retriving the description of the route in this specific locale.
                break
        else:
            description = locales[0].get("description") # If no locale corresponds to the desired language, we take the description of the first available locale.

    return {
        "description": description,
        "rock_types": join_or_none(data.get("rock_types")),
        "configuration": join_or_none(data.get("configuration")),
    }


# Formatting a row of data from a JSON document from the API (one way). Each row of the resulting CSV is formatted correctly. 
def parse_route_listing(doc):
    """Extrait les champs structurés d'un document de listing (/routes)."""
    row = {field: None for field in FIELDS} # Initialize a `row` dictionary with all the keys defined in `FIELDS` and initial values set to `None`. 
                                            # This ensures that all columns in the final CSV will be present, even if some information is missing in the listing document.
    # Retrieving first all the universal data (numbers or text that are internal codes: quality, climbing_outdoor_type...)
    row["document_id"] = doc.get("document_id")
    row["quality"] = doc.get("quality")
    row["activities"] = ";".join(doc.get("activities") or [])
    row["climbing_outdoor_type"] = doc.get("climbing_outdoor_type")
    row["global_rating"] = doc.get("global_rating")
    row["engagement_rating"] = doc.get("engagement_rating")
    row["risk_rating"] = doc.get("risk_rating")
    row["equipment_rating"] = doc.get("equipment_rating")
    row["exposition_rock_rating"] = doc.get("exposition_rock_rating")
    row["rock_free_rating"] = doc.get("rock_free_rating")
    row["rock_required_rating"] = doc.get("rock_required_rating")
    row["aid_rating"] = doc.get("aid_rating")
    row["orientations"] = ";".join(doc.get("orientations") or [])
    row["elevation_min"] = doc.get("elevation_min")
    row["elevation_max"] = doc.get("elevation_max")
    row["height_diff_up"] = doc.get("height_diff_up")
    row["height_diff_down"] = doc.get("height_diff_down")
    row["height_diff_difficulties"] = doc.get("height_diff_difficulties")
    row["durations"] = ";".join(doc.get("durations") or [])

    # Retrieving then the translated data (localized)
    locales = doc.get("locales") or []
    if locales:
        row["title"] = locales[0].get("title")
        row["title_prefix"] = locales[0].get("title_prefix")
    return row

Fonctions définies (version corrigée : rock_types/configuration lus depuis le détail).


### 2.2 Scraping main fuction

In [ ]:
# Main scraping loop: paginates through the listing, retrieves the details, and writes the results to a CSV file as it goes.
def scrape(max_routes, output_path, fetch_descriptions=True, verbose=True):

    offset = 0
    total_written = 0
    total_available = None
    all_rows = []

    # Open the CSV file in write mode, using UTF-8 encoding and without any extra blank lines between rows.
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()

        while total_written < max_routes:
            page_limit = min(PAGE_SIZE, max_routes - total_written) # calculate the number of routes to fetch for this page, taking the minimum between PAGE_SIZE and the number of remaining routes to fetch (max_routes - total_written).
            if verbose: # very common parameter to display or not progress messages in the console. If verbose is True, we display the following message.
                print(f"[Listing] offset={offset}, limit={page_limit}...")

            data = fetch_routes_page(offset=offset, limit=page_limit) # fetch a page of climbing routes from the API by calling the fetch_routes_page function with the previously calculated offset and limit parameters.
            documents = data.get("documents", []) # retrieve the list of documents (climbing routes) from the dictionary data returned by the API. If the key "documents" does not exist, we get an empty list by default.
            total_available = data.get("total", total_available) # retrieve the total number of climbing routes available from the API response. If the key "total" does not exist, we keep the previous value of total_available.

            if not documents:
                print("No more routes to fetch.")
                break

            for doc in documents:
                row = parse_route_listing(doc) # transform each climbing route document into a formatted CSV row by calling the parse_route_listing function.
                if fetch_descriptions: # if the fetch_descriptions parameter is True, retrieve the detailed information for each climbing route (description, rock types, configuration) by calling the fetch_route_detail function with the route's document_id.
                    try:
                        detail = fetch_route_detail(row["document_id"])
                        row["description"] = detail["description"]
                        row["rock_types"] = detail["rock_types"]
                        row["configuration"] = detail["configuration"]
                    except requests.RequestException as e:
                        print(f"  ! Error fetching details for {row['document_id']}: {e}")
                    time.sleep(SLEEP_BETWEEN_CALLS)

                writer.writerow(row) # write the formatted data row to the CSV file using the previously created writer object. Each row corresponds to a climbing route.
                all_rows.append(row) # add the formatted data row to the all_rows list, which contains all the rows of data retrieved and formatted so far. This allows keeping a copy of the data in memory for further processing if needed.
                total_written += 1
                if total_written >= max_routes:
                    break

            if verbose:
                print(f"  -> {total_written} routes written (total available: {total_available})")
            offset += page_limit
            time.sleep(SLEEP_BETWEEN_CALLS)

            if total_available is not None and offset >= total_available:
                break

    print(f"\nTerminé. {total_written} routes written in {output_path}")
    return all_rows

## 3. Quick Test (10 fields)

Run this first. It should take less than 15 seconds.
Verifies that the API is responding correctly and that the table below looks correct
(columns filled in, descriptions not empty).

In [ ]:
rows_test = scrape(max_routes=10, output_path="test_10.csv", fetch_descriptions=True)

df_test = pd.DataFrame(rows_test)
df_test


## 4. Intermediate Test (500 tracks)

Run only if the previous step went well.
Estimated duration: about 5 to 10 minutes.

In [ ]:
rows_500 = scrape(max_routes=500, output_path="test_500.csv", fetch_descriptions=True)

df_500 = pd.DataFrame(rows_500)
df_500.head(20)


In [ ]:
# A few quick checks on the sample of 500 tracks

print("Number of routes retrieved :", len(df_500))
print("\nMissing values per column :")
print(df_500.isna().sum())
print("\nExample of description :")
print(df_500['description'].dropna().iloc[0])


## 5. Retrieving data from a sample (< 10,000 channels)

⚠️ This cell uses `scrape()`, which simply paginates by `offset`.
**It CANNOT exceed 10,000 channels** (API limit; see Section 7
to understand and work around this). Use it only for a
deliberately limited sample (e.g., 5,000 channels); to retrieve everything,
go directly to **Section 7**.

In [ ]:
max_routes = 5000 # VOLUNTARILY remains below 10,000 (API limit; see Section 7)

rows_sample = scrape(max_routes=max_routes, output_path="camptocamp_routes_sample.csv", fetch_descriptions=True)

df_sample = pd.DataFrame(rows_sample)
print("Number of routes retrieved :", len(df_sample))
df_sample.head(20)


## 6. Full Retrieval — Bypassing the 10,000-result Limit

**Important**: The API rejects any request with `offset >= 10000`
(`400 Bad Request` error). This is a technical limitation of the
search engine (Elasticsearch) used by Camptocamp, not a bug in our code.

**Solution**: We automatically divide the world into geographic zones
(using the `bbox` parameter), recursively subdividing any zone that
still contains more than 9,000 routes, until each zone is
below the limit. We can then paginate normally within each zone.

This section also automatically resumes where it left off in the event
of a crash: it re-reads the routes already present in the CSV file and
ignores them.## 6. Full retrieval — workaround for the 10,000-result limit

### 6.1 defining side fuctions

In [ ]:
# The Camptocamp API has a server-side bug: its bbox filter is silently ignored if any of the four coordinates is exactly 0 (in Python, `all([...])`
# returns False if any value is 0). Since our initial bbox ranges from longitude -180 to +180, the very first slice falls exactly on x=0 (the
# Greenwich Meridian), which breaks the filter for an entire section of the binary search tree. We fix this by shifting very slightly (1 mm, in
# meters) any coordinate that would be exactly 0—without any real impact on geographic accuracy.

BBOX_EPSILON = 1e-3

# Avoiding using zero coordinates to work around the API bug (the bbox filter is silently ignored if any of the four coordinates is exactly 0)
def avoid_zero(v):
    return BBOX_EPSILON if v == 0 else v

# Converts longitude/latitude coordinates (degrees) to Web Mercator (EPSG:3857, meters) so that the bbox parameter of the Camptocamp API works correctly.
def lonlat_to_mercator(lon, lat):
    x = lon * 20037508.34 / 180.0
    y = math.log(math.tan((90 + lat) * math.pi / 360.0)) / (math.pi / 180.0)
    y = y * 20037508.34 / 180.0
    return avoid_zero(x), avoid_zero(y)

# Calls the previous function to convert multiple coordinates at once: the lower-left corner and the upper-right corner that define a bbox.
def lonlat_bbox_to_mercator(lon_min, lat_min, lon_max, lat_max):
    x_min, y_min = lonlat_to_mercator(lon_min, lat_min)
    x_max, y_max = lonlat_to_mercator(lon_max, lat_max)
    return (x_min, y_min, x_max, y_max)

# Fetches a page of the list of climbing routes, in a certain geographic area (with an optional bbox parameter but used in our case).
def fetch_routes_page(offset, limit=PAGE_SIZE, bbox=None):
    params = {"act": "rock_climbing", "limit": limit, "offset": offset, "pl": LANG}
    if bbox is not None:
        params["bbox"] = ",".join(str(v) for v in bbox)
    resp = requests.get(f"{BASE_URL}/routes", params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Returns the total number of routes in a given area (1 single lightweight call).
def get_bbox_count(bbox):
    data = fetch_routes_page(offset=0, limit=1, bbox=bbox)
    return data.get("total", 0)

# Recursively splits a bounding box into 4 quadrants as long as the number of lanes it contains exceeds max_count.
def find_leaf_bboxes(bbox, max_count=9000, depth=0, max_depth=12, verbose=True):
    total = get_bbox_count(bbox)
    if total == 0:
        return []
    if total <= max_count or depth >= max_depth:
        if verbose:
            print(f"  zone retenue : {total} voies (profondeur {depth})")
        return [(bbox, total)]

    x_min, y_min, x_max, y_max = bbox
    x_mid, y_mid = avoid_zero((x_min + x_max) / 2), avoid_zero((y_min + y_max) / 2)
    quadrants = [
        (x_min, y_min, x_mid, y_mid),
        (x_mid, y_min, x_max, y_mid),
        (x_min, y_mid, x_mid, y_max),
        (x_mid, y_mid, x_max, y_max),
    ]
    leaves = []
    for q in quadrants:
        leaves.extend(find_leaf_bboxes(q, max_count, depth + 1, max_depth, verbose))
    return leaves


Fonctions de récupération par découpage géographique définies.


### 6.2 Scraping main fuction

In [ ]:
# Comprehensive data scraping by geographic region, with automatic resumption in case of a retry.

def scrape_via_bbox(output_path, fetch_descriptions=True, max_count_per_leaf=9000, verbose=True):
    # 1. Load previously retrieved tracks (for resuming or deduplication)
    processed_ids = set()
    file_exists = os.path.exists(output_path)
    if file_exists:
        with open(output_path, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                processed_ids.add(row["document_id"])
        print(f"{len(processed_ids)} unique routes already present in {output_path} (will be ignored).")

    # 2. Divide the world into zones below the threshold
    print("Geographic partitioning in progress...")
    world_bbox = lonlat_bbox_to_mercator(-180, -60, 180, 85)
    leaves = find_leaf_bboxes(world_bbox, max_count=max_count_per_leaf, verbose=verbose)
    print(f"\n{len(leaves)} zones to explore, {sum(t for _, t in leaves)} routes in total (possible duplicate borders, automatically deduplicated).\n")

    mode = "a" if file_exists else "w"
    total_written = len(processed_ids)

    with open(output_path, mode, newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if not file_exists:
            writer.writeheader()

        for zone_num, (leaf_bbox, leaf_total) in enumerate(leaves, start=1):
            offset = 0
            while offset < leaf_total:
                limit = min(PAGE_SIZE, leaf_total - offset)
                try:
                    data = fetch_routes_page(offset=offset, limit=limit, bbox=leaf_bbox)
                except requests.RequestException as e:
                    print(f"  ! Zone error {zone_num}, offset {offset}: {e}")
                    break

                documents = data.get("documents", [])
                if not documents:
                    break

                for doc in documents:
                    doc_id = str(doc.get("document_id"))
                    if doc_id in processed_ids:
                        continue  # already retrieved during a previous execution or in an adjacent area

                    row = parse_route_listing(doc)
                    if fetch_descriptions:
                        try:
                            detail = fetch_route_detail(row["document_id"])
                            row["description"] = detail["description"]
                            row["rock_types"] = detail["rock_types"]
                            row["configuration"] = detail["configuration"]
                        except requests.RequestException as e:
                            print(f"  ! Error fetching details for {row['document_id']}: {e}")
                        time.sleep(SLEEP_BETWEEN_CALLS)

                    writer.writerow(row)
                    f.flush()
                    processed_ids.add(doc_id)
                    total_written += 1

                offset += limit
                time.sleep(SLEEP_BETWEEN_CALLS)

            if verbose:
                print(f"[{zone_num}/{len(leaves)}] zone done ({leaf_total} routes in the zone). Cumulative written total : {total_written}")

    print(f"\Finished. {total_written} unique routes written to {output_path}")
    return total_written

### 6.3 Scraping data

In [ ]:
scrape_via_bbox(output_path="camptocamp_routes_full.csv", fetch_descriptions=True)

Découpage géographique en cours...
  zone retenue : 1605 voies (profondeur 1)
  zone retenue : 70 voies (profondeur 2)
  zone retenue : 15 voies (profondeur 2)
  zone retenue : 13 voies (profondeur 3)
  zone retenue : 15 voies (profondeur 3)
  zone retenue : 47 voies (profondeur 4)
  zone retenue : 111 voies (profondeur 4)
  zone retenue : 159 voies (profondeur 5)
  zone retenue : 140 voies (profondeur 5)
  zone retenue : 1739 voies (profondeur 6)
  zone retenue : 677 voies (profondeur 6)
  zone retenue : 3141 voies (profondeur 6)
  zone retenue : 5774 voies (profondeur 6)
  zone retenue : 132 voies (profondeur 5)
  zone retenue : 54 voies (profondeur 4)
  zone retenue : 66 voies (profondeur 3)
  zone retenue : 36 voies (profondeur 2)
  zone retenue : 283 voies (profondeur 1)
  zone retenue : 2822 voies (profondeur 1)

19 zones à parcourir, 16899 voies au total (doublons de bord possibles, dédoublonnés automatiquement).

[Zone 1/19] terminée (1605 voies dans la zone). Total cumulé écri

16860

## 7. Data loading

In [9]:
df = pd.read_csv("camptocamp_routes_full.csv")
df

,document_id,title,title_prefix,quality,activities,climbing_outdoor_type,global_rating,engagement_rating,risk_rating,equipment_rating,...,rock_types,orientations,configuration,elevation_min,elevation_max,height_diff_up,height_diff_down,height_diff_difficulties,durations,description
0,1948187,Triangle des Coutchets - Brouillard d'été,Pic des Coutchets,draft,rock_climbing,multi,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A gauche de Aquim
1,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,calcaire,NE,face,NaN,NaN,NaN,NaN,150.0,1,Approche: monter au plateau de Bezou soit à pi...
2,1944609,Capricho,Aiguille de la Canalona,medium,rock_climbing,multi,D+,NaN,NaN,P2+,...,calcaire,S,pillar,1850.0,2540.0,690.0,NaN,170.0,1,"## Approche ##\n\nDepuis le Câble, on suit la ..."
3,1944282,Secteur Rocaïne - Cash cache,Les Douanes,fine,rock_climbing,single,NaN,NaN,NaN,NaN,...,calcaire,S,NaN,1560.0,1610.0,100.0,100.0,15.0,1,## Approche # 15min\nDu parking en bordure de ...
4,1944020,Face W - Heureux qui comme Ulysse,Pène Sarrière,fine,rock_climbing,multi,ED-,I,X2,P1,...,calcaire,W,face,1370.0,1830.0,NaN,NaN,195.0,1,## Approche # 45min\nSuivre l'approche de la [...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16855,54116,L'avenue des chamois,Motteux Péteux,fine,rock_climbing,multi,AD+,II,NaN,P1+,...,calcaire,S,face,1633.0,2232.0,680.0,680.0,300.0,1,"## Approche # 1h15\nDe Mittelberg (1632 m), su..."
16856,53904,Voie Originale de l'Y,Miroir d'Argentine,medium,rock_climbing,multi,D-,II,NaN,P1+,...,NaN,N,face,1469.0,2325.0,850.0,892.0,380.0,1,"## Approche # (env. 1h)\nDu parking supérieur,..."
16857,53903,Papageno,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,NaN,NW,face,1462.0,2325.0,800.0,887.0,400.0,1,"## Approche\nDu parking, traverser le torrent ..."
16858,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,calcaire,NW,face,1462.0,2325.0,800.0,800.0,400.0,1,[img=1045456 right]Tracé de la voie[/img]\n\n#...
